# 06 — Sodar et al. (2026): Angle vs Amplitude QSVM, transferred to 4 diabetes datasets

**Method (EPJ Quantum Technol.):** QSVM with angle encoding and with amplitude encoding,
originally on URL-traffic data (70/30 split, N = 3,000). That data is not diabetes, so only the
*method* is transferred here.

**Protocols**
* `paper` — 70/30 random stratified split (seed 42), ungrouped, scaling fitted on all rows.
* `clean` — leakage-free grouped/stratified 5-fold × seeds {0, 1, 42}, in-fold preprocessing.

Amplitude encoding is run on two inputs: MinMax [0, π] features (as in the paper — every vector
is non-negative, so the kernel only sees direction) and standardised features (the repair used in
the audit). RBF-SVM and LR are the classical references.

In [1]:
import os, sys
from pathlib import Path
for p in [Path.cwd(), Path.cwd() / "notebooks2", Path("/content/notebooks2"), *Path("/kaggle/input").glob("*"), *Path("/kaggle/input").glob("*/notebooks2")]:
    if (p / "nb2_common.py").exists():
        sys.path.insert(0, str(p)); break
from nb2_common import *
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
sanity_check_kernels()
DATA = load_all()

nb2_common loaded: numpy 2.4.6, sklearn 1.6.1
kernel self-test passed
pima     N=  768 d=  8 prevalence=0.349 unique=  768 grouped_cv=False  [diabetes]
sylhet   N=  520 d= 16 prevalence=0.615 unique=  251 grouped_cv=True  [diabetes (early stage)]


cdc      N= 2000 d= 21 prevalence=0.140 unique= 1995 grouped_cv=True  [diabetes or prediabetes (self-reported)]


hosp130  N= 2000 d= 44 prevalence=0.088 unique= 2000 grouped_cv=False  [readmitted within 30 days]


In [2]:
def sodar_method(Xtr, ytr, Xte, seed, groups_tr, X_prep):
    Xq_tr, Xq_te = prep(to_qubits, X_prep, Xtr, Xte, seed=seed)
    Xs_tr, Xs_te = prep(impute_scale, X_prep, Xtr, Xte)
    Xm_tr, Xm_te = prep(impute_scale, X_prep, Xtr, Xte, kind="minmax")
    out = {
        "Angle-QSVM": svm_precomputed(angle_kernel(Xq_tr, Xq_tr), ytr, angle_kernel(Xq_te, Xq_tr), seed=seed),
        "Amplitude-QSVM (MinMax, paper)": svm_precomputed(amplitude_kernel(Xm_tr, Xm_tr), ytr,
                                                          amplitude_kernel(Xm_te, Xm_tr), seed=seed),
        "Amplitude-QSVM (standardised)": svm_precomputed(amplitude_kernel(Xs_tr, Xs_tr), ytr,
                                                         amplitude_kernel(Xs_te, Xs_tr), seed=seed),
    }
    rbf = SVC(kernel="rbf", C=1.0, gamma="scale", random_state=seed).fit(Xs_tr, ytr)
    out["RBF-SVM"] = (rbf.predict(Xs_te), rbf.decision_function(Xs_te))
    lr = LogisticRegression(max_iter=1000).fit(Xs_tr, ytr)
    out["Logistic Regression"] = (lr.predict(Xs_te), lr.predict_proba(Xs_te)[:, 1])
    out["Majority class"] = (np.full(len(Xte), int(ytr.mean() >= 0.5)), np.zeros(len(Xte)))
    return out


rec = Recorder("P06_Sodar", "06_sodar", config=dict(
    method="Angle-QSVM, Amplitude-QSVM on MinMax and on standardised input, RBF-SVM, LR",
    paper_protocol="70/30 stratified hold-out, seed 42, global scaling"))
print("paper protocol"); run_paper_holdout(rec, DATA, sodar_method, test_size=0.3, global_prep=True)
print("clean protocol"); run_clean(rec, DATA, sodar_method)
df = rec.save()

paper protocol


  paper  pima     test rows sharing a feature vector with train: 0


  paper  sylhet   test rows sharing a feature vector with train: 106


  paper  cdc      test rows sharing a feature vector with train: 2


  paper  hosp130  test rows sharing a feature vector with train: 0
clean protocol


  clean  pima     done in    3.4s


  clean  sylhet   done in    2.7s


  clean  cdc      done in   13.7s


  clean  hosp130  done in   16.0s
saved /content/notebooks2/results/06_sodar.csv  (384 rows)


In [3]:
pd.set_option("display.width", 220, "display.max_rows", 200)
summarise(df)

,dataset,protocol,model,accuracy,balanced_accuracy,recall,specificity,roc_auc,n_runs
0,cdc,clean,"Amplitude-QSVM (MinMax, paper)",0.86 ± 0.018,0.5 ± 0.0,0.0 ± 0.0,1.0 ± 0.0,0.769 ± 0.027,15
1,cdc,clean,Amplitude-QSVM (standardised),0.86 ± 0.018,0.5 ± 0.001,0.0 ± 0.0,1.0 ± 0.001,0.692 ± 0.031,15
2,cdc,clean,Angle-QSVM,0.86 ± 0.018,0.5 ± 0.0,0.0 ± 0.0,1.0 ± 0.0,0.717 ± 0.034,15
3,cdc,clean,Logistic Regression,0.863 ± 0.017,0.577 ± 0.018,0.18 ± 0.039,0.974 ± 0.006,0.818 ± 0.02,15
4,cdc,clean,Majority class,0.86 ± 0.018,0.5 ± 0.0,0.0 ± 0.0,1.0 ± 0.0,0.5 ± 0.0,15
5,cdc,clean,RBF-SVM,0.862 ± 0.017,0.531 ± 0.018,0.071 ± 0.039,0.99 ± 0.005,0.75 ± 0.037,15
6,cdc,paper,"Amplitude-QSVM (MinMax, paper)",0.86 ± nan,0.5 ± nan,0.0 ± nan,1.0 ± nan,0.764 ± nan,1
7,cdc,paper,Amplitude-QSVM (standardised),0.86 ± nan,0.5 ± nan,0.0 ± nan,1.0 ± nan,0.711 ± nan,1
8,cdc,paper,Angle-QSVM,0.86 ± nan,0.5 ± nan,0.0 ± nan,1.0 ± nan,0.762 ± nan,1
9,cdc,paper,Logistic Regression,0.855 ± nan,0.552 ± nan,0.131 ± nan,0.973 ± nan,0.824 ± nan,1


## Does Amplitude (MinMax) collapse to one class? (`n_pred_pos` = predicted positives per fold)

In [4]:
df.groupby(["dataset", "protocol", "model"])[["n_pred_pos", "n_pos_test", "recall", "specificity"]].mean().round(3)

n_pred_pos  n_pos_test  recall  specificity
dataset protocol model                                                                      
cdc     clean    Amplitude-QSVM (MinMax, paper)       0.000        55.8   0.000        1.000
                 Amplitude-QSVM (standardised)        0.133        55.8   0.000        1.000
                 Angle-QSVM                           0.000        55.8   0.000        1.000
                 Logistic Regression                 18.933        55.8   0.180        0.974
                 Majority class                       0.000        55.8   0.000        1.000
                 RBF-SVM                              7.267        55.8   0.071        0.990
        paper    Amplitude-QSVM (MinMax, paper)       0.000        84.0   0.000        1.000
                 Amplitude-QSVM (standardised)        0.000        84.0   0.000        1.000
                 Angle-QSVM                           0.000        84.0   0.000        1.000
                 Logistic Regression                 25.000        84.0   0.131        0.973
                 Majority class                       0.000        84.0   0.000        1.000
                 RBF-SVM                              7.000        84.0   0.048        0.994
hosp130 clean    Amplitude-QSVM (MinMax, paper)       0.000        35.2   0.000        1.000
                 Amplitude-QSVM (standardised)        0.000        35.2   0.000        1.000
                 Angle-QSVM                           0.000        35.2   0.000        1.000
                 Logistic Regression                  1.733        35.2   0.009        0.996
                 Majority class                       0.000        35.2   0.000        1.000
                 RBF-SVM                              0.000        35.2   0.000        1.000
        paper    Amplitude-QSVM (MinMax, paper)       0.000        53.0   0.000        1.000
                 Amplitude-QSVM (standardised)        0.000        53.0   0.000        1.000
                 Angle-QSVM                           0.000        53.0   0.000        1.000
                 Logistic Regression                  5.000        53.0   0.019        0.993
                 Majority class                       0.000        53.0   0.000        1.000
                 RBF-SVM                              0.000        53.0   0.000        1.000
pima    clean    Amplitude-QSVM (MinMax, paper)      15.933        53.6   0.167        0.930
                 Amplitude-QSVM (standardised)       15.267        53.6   0.138        0.921
                 Angle-QSVM                          42.600        53.6   0.552        0.870
                 Logistic Regression                 42.467        53.6   0.571        0.881
                 Majority class                       0.000        53.6   0.000        1.000
                 RBF-SVM                             42.933        53.6   0.556        0.869
        paper    Amplitude-QSVM (MinMax, paper)      14.000        81.0   0.074        0.947
                 Amplitude-QSVM (standardised)       10.000        81.0   0.086        0.980
                 Angle-QSVM                          62.000        81.0   0.519        0.867
                 Logistic Regression                 62.000        81.0   0.519        0.867
                 Majority class                       0.000        81.0   0.000        1.000
                 RBF-SVM                             65.000        81.0   0.543        0.860
sylhet  clean    Amplitude-QSVM (MinMax, paper)      66.733        64.0   0.951        0.856
                 Amplitude-QSVM (standardised)       70.200        64.0   0.903        0.693
                 Angle-QSVM                          65.333        64.0   0.918        0.846
                 Logistic Regression                 64.333        64.0   0.914        0.857
                 Majority class                     104.000        64.0   1.000        0.000
                 RBF-SVM                          